## SRP077056

**paper:** [PMID: 30094964](https://onlinelibrary.wiley.com/doi/epdf/10.1111/ede.12264?saml_referrer) - Gene network variation and alternative paths to convergent evolution in turtles, 2018

**date, curator:** 2026-04-28, Sara Carsanaro

**notes**
* Dissected scapulae included cartilage, muscle, nerve, and connective tissues - for this reason i am annotating with UBERON:0001421 pectoral girdle region with definition: the pectoral girdle skeleton and associated soft tissue. Note that this includes both the skeletal elements and associated tissues (integument, muscle, etc).
    * for tail it is fine to annotate as just tail - definition is A tail that extends from the posterior tip of the organism to the anus, contains muscle and skeleton.
* for developmental stage: Scapulae were dissected during morphologically equiva-lent stages 20, 21, and 22 for Emydidae [Cordero & Janzen,2014](https://onlinelibrary.wiley.com/doi/10.1002/jmor.20226)
    * these are the final stages of embryogenesis  - i think that late embryonic stage is appropriate here

### annotation summary

In [33]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Dorsal scapula,UBERON:0001421,pectoral girdle region,other
3,Tail,UBERON:0007144,embryonic post-anal tail,perfect match


In [34]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Day 34 embryo,UBERON:0007220,late embryonic stage,missing child term
3,Day 38 embryo,UBERON:0007220,late embryonic stage,missing child term
12,Day 44 embryo,UBERON:0007220,late embryonic stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP077056"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 30/30 [00:31<00:00,  1.05s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [13]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1874762,SRP077056,Illumina HiSeq 2500,SRS1524445,,,,,,Dorsal scapula,Day 34 embryo,,,,,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
1,SRX1874751,SRP077056,Illumina HiSeq 2500,SRS1524445,,,,,,Dorsal scapula,Day 34 embryo,,,,,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
2,SRX1874750,SRP077056,Illumina HiSeq 2500,SRS1524445,,,,,,Dorsal scapula,Day 34 embryo,,,,,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
3,SRX1874779,SRP077056,Illumina HiSeq 2500,SRS1524446,,,,,,Tail,Day 38 embryo,,,,,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
4,SRX1874753,SRP077056,Illumina HiSeq 2500,SRS1524446,,,,,,Tail,Day 38 embryo,,,,,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
5,SRX1874752,SRP077056,Illumina HiSeq 2500,SRS1524446,,,,,,Tail,Day 38 embryo,,,,,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
6,SRX1874756,SRP077056,Illumina HiSeq 2500,SRS1524447,,,,,,Dorsal scapula,Day 34 embryo,,,,,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
7,SRX1874755,SRP077056,Illumina HiSeq 2500,SRS1524447,,,,,,Dorsal scapula,Day 34 embryo,,,,,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
8,SRX1874754,SRP077056,Illumina HiSeq 2500,SRS1524447,,,,,,Dorsal scapula,Day 34 embryo,,,,,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
9,SRX1874759,SRP077056,Illumina HiSeq 2500,SRS1524448,,,,,,Dorsal scapula,Day 38 embryo,,,,,,,85613,,,,,4,EB21,SAMN05220593,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB21,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [14]:
unique_sorted(library, "infoOrgan")

['Dorsal scapula' 'Tail']


In [15]:

# tail
library.loc[library["infoOrgan"] == "Tail", "anatId"] = "UBERON:0007144"
library.loc[library["infoOrgan"] == "Tail", "anatName"] = "embryonic post-anal tail"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Tail", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Tail", "anatBiologicalStatus"] = "not documented"

# Dorsal scapula
library.loc[library["infoOrgan"] == "Dorsal scapula", "anatId"] = "UBERON:0001421"
library.loc[library["infoOrgan"] == "Dorsal scapula", "anatName"] = "pectoral girdle region"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Dorsal scapula", "anatAnnotationStatus"] = "other"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Dorsal scapula", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1874762,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,,,,Dorsal scapula,Day 34 embryo,other,not documented,,,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
1,SRX1874751,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,,,,Dorsal scapula,Day 34 embryo,other,not documented,,,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
2,SRX1874750,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,,,,Dorsal scapula,Day 34 embryo,other,not documented,,,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
3,SRX1874779,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,,,,Tail,Day 38 embryo,perfect match,not documented,,,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
4,SRX1874753,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,,,,Tail,Day 38 embryo,perfect match,not documented,,,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
5,SRX1874752,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,,,,Tail,Day 38 embryo,perfect match,not documented,,,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
6,SRX1874756,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,,,,Dorsal scapula,Day 34 embryo,other,not documented,,,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
7,SRX1874755,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,,,,Dorsal scapula,Day 34 embryo,other,not documented,,,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
8,SRX1874754,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,,,,Dorsal scapula,Day 34 embryo,other,not documented,,,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
9,SRX1874759,SRP077056,Illumina HiSeq 2500,SRS1524448,UBERON:0001421,pectoral girdle region,,,,Dorsal scapula,Day 38 embryo,other,not documented,,,,,85613,,,,,4,EB2

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [16]:
unique_sorted(library, "infoStage")

['Day 34 embryo' 'Day 38 embryo' 'Day 44 embryo']


In [17]:
# all
library.loc[:,'stageId'] = 'UBERON:0007220'
library.loc[:,'stageName'] = 'late embryonic stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'missing child term'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1874762,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
1,SRX1874751,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
2,SRX1874750,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
3,SRX1874779,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
4,SRX1874753,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
5,SRX1874752,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
6,SRX1874756,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
7,SRX1874755,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
8,SRX1874754,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late e

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [18]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1874762,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
1,SRX1874751,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
2,SRX1874750,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,,,,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
3,SRX1874779,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
4,SRX1874753,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
5,SRX1874752,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,,,,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
6,SRX1874756,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
7,SRX1874755,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,85613,,,,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
8,SRX1874754,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERO

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [19]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq RNA Sample Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1874762,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
1,SRX1874751,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
2,SRX1874750,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
3,SRX1874779,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
4,SRX1874753,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
5,SRX1874752,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
6,SRX1874756,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,85613,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,EB20,SAMN05220592,34,,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
7,SRX1874755,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not docum

#### globin, replicates

In [20]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

    #libraryId       SRSId
0   SRX1874762  SRS1524445
1   SRX1874751  SRS1524445
2   SRX1874750  SRS1524445
3   SRX1874779  SRS1524446
4   SRX1874753  SRS1524446
5   SRX1874752  SRS1524446
6   SRX1874756  SRS1524447
7   SRX1874755  SRS1524447
8   SRX1874754  SRS1524447
9   SRX1874759  SRS1524448
10  SRX1874758  SRS1524448
11  SRX1874757  SRS1524448
13  SRX1874761  SRS1524449
14  SRX1874760  SRS1524449
12  SRX1874763  SRS1524449
15  SRX1874766  SRS1524450
16  SRX1874765  SRS1524450
17  SRX1874764  SRS1524450
18  SRX1874769  SRS1524451
19  SRX1874768  SRS1524451
20  SRX1874767  SRS1524451
21  SRX1874772  SRS1524452
22  SRX1874771  SRS1524452
23  SRX1874770  SRS1524452
24  SRX1874775  SRS1524453
25  SRX1874774  SRS1524453
26  SRX1874773  SRS1524453
28  SRX1874777  SRS1524454
27  SRX1874778  SRS1524454
29  SRX1874776  SRS1524454


/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_63509/3311601719.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dups = df[duplicateCheck].loc[:,['#libraryId', column]]


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [21]:
#library.loc[:,'sampleAge_value'] = ''
library.loc[:,'sampleAge_unit'] = 'embryonic day'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1874762,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
1,SRX1874751,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
2,SRX1874750,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
3,SRX1874779,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
4,SRX1874753,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
5,SRX1874752,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,,Stage 21,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
6,SRX1874756,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,85613,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,EB20,SAMN05220592,34,embryonic day,,,,,,Stage 20,,,28/04/2026,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
7,SRX1874755,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral gird

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [22]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-29'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1874762,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,,Stage 20,,SAC,2026-04-29,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
1,SRX1874751,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,,Stage 20,,SAC,2026-04-29,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
2,SRX1874750,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,,Stage 20,,SAC,2026-04-29,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CP20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
3,SRX1874779,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,,Stage 21,,SAC,2026-04-29,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
4,SRX1874753,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,,Stage 21,,SAC,2026-04-29,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
5,SRX1874752,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,,Stage 21,,SAC,2026-04-29,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,CPREF,,,,Stage 21,,TRANSCRIPTOMIC,size fractionation
6,SRX1874756,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,85613,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,EB20,SAMN05220592,34,embryonic day,,,,,,Stage 20,,SAC,2026-04-29,Library was generatred using Illumina TruSeq RNA-seq library kit by following the protocol in the kit,,EB20,,,,Stage 20,,TRANSCRIPTOMIC,size fractionation
7,SRX1874755,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:

#### comments

In [23]:
library.loc[:,'comment'] = 'PMID: 30094964'

#### save complete file with correct columns

In [24]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX1874762,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,PMID: 30094964,Stage 20,,SAC,2026-04-29
1,SRX1874751,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,PMID: 30094964,Stage 20,,SAC,2026-04-29
2,SRX1874750,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,PMID: 30094964,Stage 20,,SAC,2026-04-29
3,SRX1874779,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,PMID: 30094964,Stage 21,,SAC,2026-04-29
4,SRX1874753,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,PMID: 30094964,Stage 21,,SAC,2026-04-29
5,SRX1874752,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,PMID: 30094964,Stage 21,,SAC,2026-04-29
6,SRX1874756,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,85613,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,EB20,SAMN05220592,34,embryonic day,,,,,PMID: 30094964,Stage 20,,SAC,2026-04-29
7,SRX1874755,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,85613,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,EB20,SAMN05220592,34,embryonic day,,,,,PMID: 30094964,Stage 20,,SAC,2026-04-29
8,SRX1874754,SRP077056,Illumina HiSeq 2500,SRS1524447,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,85613,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,EB20,SAMN05220592,34,embryonic day,,,,,PMID: 30094964,Stage 20,,SAC,2026-04-29
9,SRX1874759,SRP077056,Illumina HiSeq 2500,SRS1524448,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 38 embryo,other,not documented,missing child term,NA,,,85613,TruSeq RNA Sample Preparation Kit,full_length,polyA,,4,EB21,SAMN05220593,38,embryonic day,,,,,PMID: 30094964,Stage 21,,SAC,2026-04-29


### experiment annotations

In [25]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP077056,Gene network variation and alternative paths to convergent evolution in turtles,"Diversification of the turtle's shell comprises remarkable phenotypic transformations. For instance, two divergent species convergently evolved shell-closing systems with shoulder blade (scapula) segments that enable coordinated movements with the shell. We expected these unusual structures to originate via similar changes in underlying gene networks, as skeletal segment formation is an evolutionarily conserved developmental process. We tested this hypothesis by comparing transcriptomes of scapula tissue across three stages of embryonic development in three emydid turtles from natural populations. We found that alternative strategies for skeletal segmentation were associated with interspecific differences in gene co-expression networks. Notably, mesenchyme homeobox 2 (MEOX2) and HOXA3-5 were central hubs driving the activity of 2,806 genes in a candidate network for scapula segmentation, albeit in only one species. Even so, scapula muscle overgrowth corresponded to the activity of similar myogenic networks in both species. This and other derived developmental processes were not observed in the third species, which displayed the ancestral (unsegmented) scapula condition. Differential gene expression tests against this reference lineage supported histological and network analyses. Our findings illustrate that molecular underpinnings of convergent evolution, including during the diversification of the atypical turtle ""body plan,"" are influenced by variation in underlying developmental processes.",SRA,,,,,,,PRJNA323738,30094964,,10.1111/ede.12264,,


#### experiment and protocol details

In [26]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

30

In [27]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP077056,Gene network variation and alternative paths to convergent evolution in turtles,"Diversification of the turtle's shell comprises remarkable phenotypic transformations. For instance, two divergent species convergently evolved shell-closing systems with shoulder blade (scapula) segments that enable coordinated movements with the shell. We expected these unusual structures to originate via similar changes in underlying gene networks, as skeletal segment formation is an evolutionarily conserved developmental process. We tested this hypothesis by comparing transcriptomes of scapula tissue across three stages of embryonic development in three emydid turtles from natural populations. We found that alternative strategies for skeletal segmentation were associated with interspecific differences in gene co-expression networks. Notably, mesenchyme homeobox 2 (MEOX2) and HOXA3-5 were central hubs driving the activity of 2,806 genes in a candidate network for scapula segmentation, albeit in only one species. Even so, scapula muscle overgrowth corresponded to the activity of similar myogenic networks in both species. This and other derived developmental processes were not observed in the third species, which displayed the ancestral (unsegmented) scapula condition. Differential gene expression tests against this reference lineage supported histological and network analyses. Our findings illustrate that molecular underpinnings of convergent evolution, including during the diversification of the atypical turtle ""body plan,"" are influenced by variation in underlying developmental processes.",SRA,total,Bgee 1K,30,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA323738,30094964,,10.1111/ede.12264,,


#### paper and xrefs

In [28]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://onlinelibrary.wiley.com/doi/epdf/10.1111/ede.12264?saml_referrer'
#experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP077056,Gene network variation and alternative paths to convergent evolution in turtles,"Diversification of the turtle's shell comprises remarkable phenotypic transformations. For instance, two divergent species convergently evolved shell-closing systems with shoulder blade (scapula) segments that enable coordinated movements with the shell. We expected these unusual structures to originate via similar changes in underlying gene networks, as skeletal segment formation is an evolutionarily conserved developmental process. We tested this hypothesis by comparing transcriptomes of scapula tissue across three stages of embryonic development in three emydid turtles from natural populations. We found that alternative strategies for skeletal segmentation were associated with interspecific differences in gene co-expression networks. Notably, mesenchyme homeobox 2 (MEOX2) and HOXA3-5 were central hubs driving the activity of 2,806 genes in a candidate network for scapula segmentation, albeit in only one species. Even so, scapula muscle overgrowth corresponded to the activity of similar myogenic networks in both species. This and other derived developmental processes were not observed in the third species, which displayed the ancestral (unsegmented) scapula condition. Differential gene expression tests against this reference lineage supported histological and network analyses. Our findings illustrate that molecular underpinnings of convergent evolution, including during the diversification of the atypical turtle ""body plan,"" are influenced by variation in underlying developmental processes.",SRA,total,Bgee 1K,30,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA323738,30094964,https://onlinelibrary.wiley.com/doi/epdf/10.1111/ede.12264?saml_referrer,10.1111/ede.12264,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [29]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [30]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [31]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [32]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [35]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
62646,SRX28917890,SRP586826,HiSeq X Ten,SRS25142669,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,adult,perfect match,not documented,perfect match,M,,,59729,,,polyA,,,CNZ12,SAMN48628562,,,,,,,"no PMID, newer data, to revisit and look for p...",control,,SAC,2026-04-28
62647,SRX28917891,SRP586826,HiSeq X Ten,SRS25142670,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,adult,perfect match,not documented,perfect match,F,,,59729,,,polyA,,,CNZ13,SAMN48628550,,,,,,,"no PMID, newer data, to revisit and look for p...",control,,SAC,2026-04-28
62648,SRX1874762,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,PMID: 30094964,Stage 20,,SAC,2026-04-29
62649,SRX1874751,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,PMID: 30094964,Stage 20,,SAC,2026-04-29
62650,SRX1874750,SRP077056,Illumina HiSeq 2500,SRS1524445,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 34 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,CP20,SAMN05220588,34,embryonic day,,,,,PMID: 30094964,Stage 20,,SAC,2026-04-29
62651,SRX1874779,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,PMID: 30094964,Stage 21,,SAC,2026-04-29
62652,SRX1874753,SRP077056,Illumina HiSeq 2500,SRS1524446,UBERON:0007144,embryonic post-anal tail,UBERON:0007220,late embryonic stage,,Tail,Day 38 embryo,perfect match,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,CPREF,SAMN05220591,38,embryonic day,,,,,PMID: 30094964,Stage 21,,SAC,2026-04-29


In [36]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1209,SRP661912,Evolutionary landscapes of zygotic genome acti...,We investigate transcriptomes of cleavage-stag...,SRA,total,Bgee 1K,138,TruSeq Stranded Total RNA,full_length,,PRJNA1402383,,https://www.biorxiv.org/content/10.64898/2026....,10.64898/2026.04.16.718233,,Mail sent April 28 2026 to mirimia@gmail.com <...
1210,SRP586826,zebra finch and budgerigar Raw sequence reads,The 16S rRNA gene and hepatic transcriptomic s...,SRA,partial,Bgee 1K,37,,,,PRJNA1094658,,,,,"no PMID, newer data, to revisit and look for p..."
1211,SRP077056,Gene network variation and alternative paths t...,Diversification of the turtle's shell comprise...,SRA,total,Bgee 1K,30,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA323738,30094964,https://onlinelibrary.wiley.com/doi/epdf/10.11...,10.1111/ede.12264,,


### add annotations to git

In [37]:
! git pull

Already up to date.


In [38]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [39]:
! git add $git_experiment_path $git_library_path

In [40]:
! git commit -m $commit_message_exp

[develop aa8d2fc] adding annotated bulk experiment SRP077056
 2 files changed, 31 insertions(+)


In [41]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.06 KiB | 783.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   349e4b7..aa8d2fc  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push